In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    roc_auc_score,
    precision_score,
    recall_score,
)

df = pd.read_csv("telecom_master.csv")
print(df.shape)
print(df["churn"].value_counts(normalize=True).round(3))

In [ ]:
df["complaint_intensity"] = df["complaints_6m"] / (df["tenure_months"] / 6).clip(
    lower=1
)
# then complaint_intensity joins num_cols the same way avg_monthly_gb etc. do

In [ ]:
target = "churn"
leaky = ["retention_call_flag", "total_charges"]  # replace with your own findings


def make_X(frame, drop):
    X = frame.drop(columns=[target] + drop + ["customer_id"])
    return X


num_cols = make_X(df, leaky).select_dtypes(include=np.number).columns.tolist()
cat_cols = make_X(df, leaky).select_dtypes(exclude=np.number).columns.tolist()
print(len(num_cols), "numeric,", len(cat_cols), "categorical")
print("complaint_intensity" in num_cols)  # should print True

In [ ]:
X = make_X(df, leaky)
y = df[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=42
)

pre = ColumnTransformer(
    [
        (
            "num",
            Pipeline(
                [("imp", SimpleImputer(strategy="median")), ("sc", StandardScaler())]
            ),
            num_cols,
        ),
        (
            "cat",
            Pipeline(
                [
                    ("imp", SimpleImputer(strategy="most_frequent")),
                    ("oh", OneHotEncoder(handle_unknown="ignore")),
                ]
            ),
            cat_cols,
        ),
    ]
)

In [ ]:
models = {
    "Logistic regression": LogisticRegression(max_iter=2000),
    "Decision tree": DecisionTreeClassifier(max_depth=6, random_state=42),
    "Random forest": RandomForestClassifier(
        n_estimators=300, min_samples_leaf=3, random_state=42
    ),
}

results = []
for name, clf in models.items():
    pipe = Pipeline([("pre", pre), ("clf", clf)]).fit(X_train, y_train)
    proba = pipe.predict_proba(X_test)[:, 1]
    pred = (proba >= 0.5).astype(int)
    results.append(
        {
            "model": name,
            "roc_auc": roc_auc_score(y_test, proba),
            "precision": precision_score(y_test, pred, zero_division=0),
            "recall": recall_score(y_test, pred),
            "accuracy": (pred == y_test).mean(),
        }
    )

comparison = pd.DataFrame(results).round(3)
comparison

In [ ]:
best = Pipeline([("pre", pre), ("clf", models["Random forest"])]).fit(X_train, y_train)
proba = best.predict_proba(X_test)[:, 1]
cm = confusion_matrix(y_test, (proba >= 0.5).astype(int))
tn, fp, fn, tp = cm.ravel()
print(f"TN {tn}  FP {fp}  FN {fn}  TP {tp}")
print(classification_report(y_test, (proba >= 0.5).astype(int), digits=3))

In [ ]:
baseline_accuracy = 1 - y_test.mean()
print("Predict nobody churns  - accuracy:", round(baseline_accuracy, 3))
print(
    "Random forest at 0.5   - accuracy:",
    round(((proba >= 0.5).astype(int) == y_test).mean(), 3),
)

In [ ]:
rows = []
for t in np.arange(0.10, 0.65, 0.05):
    pred = (proba >= t).astype(int)
    rows.append(
        {
            "threshold": round(t, 2),
            "precision": precision_score(y_test, pred, zero_division=0),
            "recall": recall_score(y_test, pred),
            "flagged": int(pred.sum()),
        }
    )
sweep = pd.DataFrame(rows).round(3)

plt.plot(sweep.threshold, sweep.precision, marker="o", label="Precision")
plt.plot(sweep.threshold, sweep.recall, marker="o", label="Recall")
plt.xlabel("Decision threshold")
plt.legend()
plt.grid(alpha=0.3)
plt.show()
sweep

In [ ]:
COST_FN, COST_FP = 5880, 300
sweep["expected_cost"] = 0
for i, t in enumerate(sweep.threshold):
    pred = (proba >= t).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, pred).ravel()
    sweep.loc[i, "expected_cost"] = fn * COST_FN + fp * COST_FP
sweep.sort_values("expected_cost").head()